<a href="https://colab.research.google.com/github/panarisivalentino-pixel/prediccion-rendimiento-agricola-argentina/blob/main/notebooks/03_union_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Join de los dos Datasets que vamos a utilizar (clima y produccion)

## Carga de datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### carga de Produccion

In [ ]:
import os

nombre_buscado = "df_produccion_clean.csv"
ruta_encontrada = None

# Buscar el archivo en Google Drive
for raiz, directorios, archivos in os.walk('/content/drive/MyDrive'):
    if nombre_buscado.lower() in [f.lower() for f in archivos]:
        for archivo in archivos:
            if archivo.lower() == nombre_buscado.lower():
                ruta_encontrada = os.path.join(raiz, archivo)
                break

if ruta_encontrada:
    print(f" ARCHIVO ENCONTRADO EN:\n{ruta_encontrada}")
else:
    print(" No se encontró el archivo con ese nombre exacto. Revisa el paso 3.")

 No se encontró el archivo con ese nombre exacto. Revisa el paso 3.


In [ ]:
import pandas as pd

# Ruta exacta donde se encontró el archivo
ruta = '/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_produccion_clean.csv'

# Cargar el archivo Excel
df_produccion = pd.read_csv(ruta)

# Visualizar las primeras filas
display(df_produccion.shape)
df_produccion.head()

(160491, 10)

,cultivo,año,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha
0,ajo,1969,Buenos Aires,6,25 de Mayo,6854,3,3,10,3333
1,ajo,1969,Buenos Aires,6,Adolfo Gonzales Chaves,6014,15,15,82,5467
2,ajo,1969,Buenos Aires,6,Almirante Brown,6028,2,2,8,4000
3,ajo,1969,Buenos Aires,6,Balcarce,6063,450,450,2025,4500
4,ajo,1969,Buenos Aires,6,Cañuelas,6134,2,2,7,3500


### Carga de Clima


In [ ]:
import os

nombre_buscado = "df_clima_clean.csv"
ruta_encontrada = None

# Buscar el archivo en Google Drive
for raiz, directorios, archivos in os.walk('/content/drive/MyDrive'):
    if nombre_buscado.lower() in [f.lower() for f in archivos]:
        for archivo in archivos:
            if archivo.lower() == nombre_buscado.lower():
                ruta_encontrada = os.path.join(raiz, archivo)
                break

if ruta_encontrada:
    print(f" ARCHIVO ENCONTRADO EN:\n{ruta_encontrada}")
else:
    print(" No se encontró el archivo con ese nombre exacto. Revisa el paso 3.")

 No se encontró el archivo con ese nombre exacto. Revisa el paso 3.


In [ ]:
import pandas as pd

# Ruta exacta donde se encontró el archivo
ruta = '/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_clima_clean.csv'

# Cargar el archivo Excel
df_clima = pd.read_csv(ruta)

# Visualizar las primeras filas
display(df_clima.shape)
df_clima.head()

(1035, 9)

,provincia,año,T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,precip_total_mm,meses_con_dato
0,Buenos Aires,1981,15.38,28.65,2.54,0.00,71.83,805.9,12
1,Buenos Aires,1982,15.31,29.05,2.75,0.00,71.59,844.6,12
2,Buenos Aires,1983,14.78,28.08,2.99,0.00,73.14,908.3,12
3,Buenos Aires,1984,14.35,27.18,2.35,16.69,75.99,924.4,12
4,Buenos Aires,1985,15.56,28.73,2.98,17.59,70.13,809.9,12


## Union de datasets

In [ ]:
import pandas as pd

df = df_produccion.merge(df_clima, on=['provincia', 'año'], how='left')

print(df.shape)  # (160491, 17)
print(df.isna().sum()[df.isna().sum() > 0])  # columnas climáticas con NaN (años sin dato de clima)

(160491, 17)
T2M                  49074
T2M_MAX              49074
T2M_MIN              49074
ALLSKY_SFC_SW_DWN    49074
RH2M                 49074
precip_total_mm      49074
meses_con_dato       49074
dtype: int64


## Recorte y procesamiento del nuevo dataset dataset

In [ ]:
# Opción A: solo años con clima real (recortado)
df_recortado = df.dropna(subset=['T2M'])

# Opción B: todos los años, imputando climáticas faltantes con la media por provincia
cols_clima = ['T2M', 'T2M_MAX', 'T2M_MIN', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'precip_total_mm', 'meses_con_dato']

df_completo = df.copy()
df_completo['tiene_dato_clima'] = df_completo['T2M'].notna().astype(int)

for col in cols_clima:
    df_completo[col] = df_completo.groupby('provincia')[col].transform(lambda x: x.fillna(x.mean()))

In [ ]:
display(df_recortado.shape)
df_recortado.head()

(111417, 17)

,cultivo,año,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha,T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,precip_total_mm,meses_con_dato
2465,ajo,1981,Buenos Aires,6,Bahía Blanca,6056,10,10,39,3900,15.38,28.65,2.54,0.0,71.83,805.9,12.0
2466,ajo,1981,Buenos Aires,6,Balcarce,6063,275,275,990,3600,15.38,28.65,2.54,0.0,71.83,805.9,12.0
2467,ajo,1981,Buenos Aires,6,Coronel de Marina Leonardo Rosales,6182,10,10,38,3800,15.38,28.65,2.54,0.0,71.83,805.9,12.0
2468,ajo,1981,Buenos Aires,6,Coronel Pringles,6196,4,4,18,4500,15.38,28.65,2.54,0.0,71.83,805.9,12.0
2469,ajo,1981,Buenos Aires,6,General Pueyrredón,6357,3,3,10,3333,15.38,28.65,2.54,0.0,71.83,805.9,12.0


In [ ]:
display(df_completo.shape)
df_completo.head()

(160491, 18)

,cultivo,año,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha,T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,precip_total_mm,meses_con_dato,tiene_dato_clima
0,ajo,1969,Buenos Aires,6,25 de Mayo,6854,3,3,10,3333,14.941513,28.471547,2.678746,15.838139,72.249857,957.310375,12.0,0
1,ajo,1969,Buenos Aires,6,Adolfo Gonzales Chaves,6014,15,15,82,5467,14.941513,28.471547,2.678746,15.838139,72.249857,957.310375,12.0,0
2,ajo,1969,Buenos Aires,6,Almirante Brown,6028,2,2,8,4000,14.941513,28.471547,2.678746,15.838139,72.249857,957.310375,12.0,0
3,ajo,1969,Buenos Aires,6,Balcarce,6063,450,450,2025,4500,14.941513,28.471547,2.678746,15.838139,72.249857,957.310375,12.0,0
4,ajo,1969,Buenos Aires,6,Cañuelas,6134,2,2,7,3500,14.941513,28.471547,2.678746,15.838139,72.249857,957.310375,12.0,0


In [ ]:
df_completo.isnull().sum()

,0
cultivo,0
año,0
provincia,0
provincia_id,0
departamento,0
departamento_id,0
superficie_sembrada_ha,0
superficie_cosechada_ha,0
produccion_tm,0
rendimiento_kgxha,0


In [ ]:
df_recortado.isnull().sum()

,0
cultivo,0
año,0
provincia,0
provincia_id,0
departamento,0
departamento_id,0
superficie_sembrada_ha,0
superficie_cosechada_ha,0
produccion_tm,0
rendimiento_kgxha,0


### Resultado
`df_recortado`: Elimina todas las filas con NaN en clima. Resultado: 111.417 filas × 17 columnas, cubriendo solo 1981-2024. Datos climáticos 100% reales, sin nada inventado. Se pierden ~30% de las filas originales.

`df_completo` : `df_completo['tiene_dato_clima']`crea una columna bandera (1 = dato climático real, 0 = imputado) antes de tocar nada, para no perder el rastro de qué se inventó.
El `for` recorre las 7 columnas climáticas y, para cada una, agrupa por provincia y rellena los NaN con el promedio histórico de esa misma provincia (no un promedio global, para que sea más realista).

## Guardamos los dos Datasets


In [ ]:
df_recortado.to_csv('/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_recortado.csv', index=False)
df_completo.to_csv('/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_completo.csv', index=False)
